# ONNX for Computer Vision — Deep Dive

This notebook explores the mathematical foundations of computer vision models
deployed with ONNX, covering convolutional neural networks, object detection
architectures (YOLO, SSD), and optimization strategies for CV inference.

## 1. Convolutional Neural Networks Overview

CNNs exploit spatial locality through weight sharing and local connectivity:

```
┌──────────────────────────────────────────────────────────────────┐
│                CNN ARCHITECTURE OVERVIEW                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                    │
│  Input Image                                                       │
│  [B, 3, 224, 224]                                                 │
│       │                                                            │
│       ▼                                                            │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐           │
│  │ Conv + BN     │  │ Conv + BN     │  │ Conv + BN     │           │
│  │ + ReLU        │──▶│ + ReLU        │──▶│ + ReLU        │           │
│  │ [B,64,112,112]│  │ [B,128,56,56] │  │ [B,256,28,28] │           │
│  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘           │
│         │ MaxPool          │ MaxPool          │ MaxPool           │
│         ▼                  ▼                  ▼                    │
│  [B,64,56,56]       [B,128,28,28]      [B,256,14,14]             │
│                                              │                    │
│                                              ▼                    │
│                                    ┌──────────────────┐          │
│                                    │  Global Avg Pool  │          │
│                                    │  [B, 256, 1, 1]   │          │
│                                    └────────┬─────────┘          │
│                                             │                     │
│                                             ▼                     │
│                                    ┌──────────────────┐          │
│                                    │  FC → Softmax     │          │
│                                    │  [B, num_classes]  │          │
│                                    └──────────────────┘          │
└──────────────────────────────────────────────────────────────────┘
```

## 2. Conv2D Mathematical Foundation

The 2D convolution operation computes:

$$Y_{n,c_{out},h,w} = \sum_{k=0}^{C_{in}-1} \sum_{r=0}^{K_h-1} \sum_{s=0}^{K_w-1} W_{c_{out},k,r,s} \cdot X_{n,k,h \cdot s_h + r,\, w \cdot s_w + s} + b_{c_{out}}$$

Where:
- $X \in \mathbb{R}^{N \times C_{in} \times H_{in} \times W_{in}}$ — Input tensor
- $W \in \mathbb{R}^{C_{out} \times C_{in} \times K_h \times K_w}$ — Filter weights
- $b \in \mathbb{R}^{C_{out}}$ — Bias
- $s_h, s_w$ — Stride
- $K_h, K_w$ — Kernel size

### Output Size Formula

$$H_{out} = \left\lfloor \frac{H_{in} + 2p_h - d_h(K_h - 1) - 1}{s_h} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{W_{in} + 2p_w - d_w(K_w - 1) - 1}{s_w} \right\rfloor + 1$$

Where $p$ = padding, $d$ = dilation, $s$ = stride.

### Common Configurations

| Config | Kernel | Stride | Padding | Output Size |
|--------|--------|--------|---------|-------------|
| Same | 3×3 | 1 | 1 | $H_{in} \times W_{in}$ |
| Downsample | 3×3 | 2 | 1 | $\lceil H_{in}/2 \rceil \times \lceil W_{in}/2 \rceil$ |
| 1×1 (pointwise) | 1×1 | 1 | 0 | $H_{in} \times W_{in}$ |
| Large kernel | 7×7 | 2 | 3 | $\lceil H_{in}/2 \rceil \times \lceil W_{in}/2 \rceil$ |

In [ ]:
import numpy as np

def conv2d_naive(input_tensor, kernel, bias=None, stride=1, padding=0):
    """Pure NumPy 2D convolution for educational purposes."""
    N, C_in, H_in, W_in = input_tensor.shape
    C_out, C_in_k, K_h, K_w = kernel.shape
    
    # Apply padding
    if padding > 0:
        input_tensor = np.pad(input_tensor,
            ((0,0), (0,0), (padding, padding), (padding, padding)))
    
    _, _, H_padded, W_padded = input_tensor.shape
    H_out = (H_padded - K_h) // stride + 1
    W_out = (W_padded - K_w) // stride + 1
    
    output = np.zeros((N, C_out, H_out, W_out))
    
    for n in range(N):
        for c_out in range(C_out):
            for h in range(H_out):
                for w in range(W_out):
                    h_start = h * stride
                    w_start = w * stride
                    receptive_field = input_tensor[n, :, 
                        h_start:h_start+K_h, w_start:w_start+K_w]
                    output[n, c_out, h, w] = np.sum(receptive_field * kernel[c_out])
            if bias is not None:
                output[n, c_out] += bias[c_out]
    
    return output

# Demonstrate
np.random.seed(42)
x = np.random.randn(1, 3, 8, 8).astype(np.float32)
w = np.random.randn(16, 3, 3, 3).astype(np.float32)
b = np.random.randn(16).astype(np.float32)

output = conv2d_naive(x, w, b, stride=1, padding=1)
print(f"Input shape:  {x.shape}  (N, C_in, H, W)")
print(f"Kernel shape: {w.shape}  (C_out, C_in, K_h, K_w)")
print(f"Output shape: {output.shape}  (N, C_out, H_out, W_out)")
print(f"\nOutput size formula: H_out = (8 + 2×1 - 3) / 1 + 1 = {(8 + 2 - 3)//1 + 1}")

## 3. Conv2D FLOPs Analysis

The computational cost of a convolution layer:

$$\text{FLOPs} = 2 \cdot C_{out} \cdot C_{in} \cdot K_h \cdot K_w \cdot H_{out} \cdot W_{out}$$

(Factor of 2 accounts for multiply-add)

$$\text{Parameters} = C_{out} \cdot C_{in} \cdot K_h \cdot K_w + C_{out}$$

### Depthwise Separable Convolution (MobileNet)

Standard conv FLOPs: $2 \cdot C_{out} \cdot C_{in} \cdot K^2 \cdot H \cdot W$

Depthwise + Pointwise: $2 \cdot C_{in} \cdot K^2 \cdot H \cdot W + 2 \cdot C_{out} \cdot C_{in} \cdot H \cdot W$

Savings ratio: $\frac{1}{C_{out}} + \frac{1}{K^2}$ (typically 8-9× fewer FLOPs)

In [ ]:
def conv_flops(C_in, C_out, K, H_out, W_out, groups=1):
    """Calculate FLOPs for a convolution layer."""
    return 2 * (C_in // groups) * C_out * K * K * H_out * W_out

def conv_params(C_in, C_out, K, groups=1):
    """Calculate parameters for a convolution layer."""
    return (C_in // groups) * C_out * K * K + C_out

# ResNet-50 layer analysis
resnet50_layers = [
    # (name, C_in, C_out, K, H_out, W_out, groups)
    ('conv1 (7×7/2)', 3, 64, 7, 112, 112, 1),
    ('res2a_1 (1×1)', 64, 64, 1, 56, 56, 1),
    ('res2a_2 (3×3)', 64, 64, 3, 56, 56, 1),
    ('res2a_3 (1×1)', 64, 256, 1, 56, 56, 1),
    ('res3a_2 (3×3)', 128, 128, 3, 28, 28, 1),
    ('res4a_2 (3×3)', 256, 256, 3, 14, 14, 1),
    ('res5a_2 (3×3)', 512, 512, 3, 7, 7, 1),
]

print("ResNet-50 Layer Analysis")
print("=" * 70)
print(f"{'Layer':<20} {'Params':>10} {'FLOPs':>12} {'Output':>14}")
print("-" * 70)

total_flops = 0
total_params = 0
for name, c_in, c_out, k, h, w, g in resnet50_layers:
    flops = conv_flops(c_in, c_out, k, h, w, g)
    params = conv_params(c_in, c_out, k, g)
    total_flops += flops
    total_params += params
    print(f"{name:<20} {params:>10,} {flops/1e6:>10.1f}M {f'[{c_out},{h},{w}]':>14}")

print("-" * 70)
print(f"{'Shown layers total':<20} {total_params:>10,} {total_flops/1e6:>10.1f}M")
print(f"\nFull ResNet-50: ~25.6M params, ~4.1 GFLOPs")

## 4. Batch Normalization

Batch Normalization normalizes activations during training:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

$$y_i = \gamma \hat{x}_i + \beta$$

During inference (and in ONNX), BatchNorm fuses with the preceding Conv:

$$W_{\text{fused}} = \frac{\gamma}{\sqrt{\sigma^2 + \epsilon}} \cdot W_{\text{conv}}$$

$$b_{\text{fused}} = \frac{\gamma}{\sqrt{\sigma^2 + \epsilon}} \cdot (b_{\text{conv}} - \mu) + \beta$$

### ONNX Fusion

```
Before Fusion:              After Fusion:
┌────────┐                  ┌────────────────────┐
│  Conv  │                  │  Conv (fused W, b) │
└───┬────┘                  └────────────────────┘
    │                           (single op!)
┌───┴──────────┐
│ BatchNorm    │
│ (γ, β, μ, σ)│
└──────────────┘
```

This fusion eliminates one memory read/write cycle per layer.

In [ ]:
import torch
import torch.nn as nn

def fuse_conv_bn(conv, bn):
    """Fuse Conv2d + BatchNorm2d into a single Conv2d."""
    fused_conv = nn.Conv2d(
        conv.in_channels, conv.out_channels, conv.kernel_size,
        stride=conv.stride, padding=conv.padding, bias=True
    )
    
    # Compute fused weights
    w_conv = conv.weight.data.clone()
    bn_weight = bn.weight.data
    bn_bias = bn.bias.data
    bn_mean = bn.running_mean
    bn_var = bn.running_var
    
    scale = bn_weight / torch.sqrt(bn_var + bn.eps)
    
    fused_conv.weight.data = w_conv * scale.reshape(-1, 1, 1, 1)
    
    if conv.bias is not None:
        fused_conv.bias.data = scale * (conv.bias.data - bn_mean) + bn_bias
    else:
        fused_conv.bias.data = scale * (-bn_mean) + bn_bias
    
    return fused_conv

# Demonstrate fusion
conv = nn.Conv2d(3, 64, 3, padding=1, bias=False)
bn = nn.BatchNorm2d(64)

# Run a batch to get running statistics
dummy = torch.randn(8, 3, 32, 32)
conv.eval()
bn.eval()
bn(conv(dummy))  # Populate running stats

fused = fuse_conv_bn(conv, bn)

# Verify equivalence
x = torch.randn(1, 3, 32, 32)
with torch.no_grad():
    original_out = bn(conv(x))
    fused_out = fused(x)

diff = (original_out - fused_out).abs().max().item()
print(f"Conv+BN Fusion Verification:")
print(f"  Original: Conv({conv.weight.shape}) + BN(64)")
print(f"  Fused:    Conv({fused.weight.shape}) with bias")
print(f"  Max difference: {diff:.2e}")
print(f"  Status: {'✓ Equivalent' if diff < 1e-5 else '✗ Mismatch'}")

## 5. ResNet Architecture and Skip Connections

ResNet solves the degradation problem with residual learning:

$$y = \mathcal{F}(x, \{W_i\}) + x$$

The identity mapping allows gradients to flow directly:

$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \left(1 + \frac{\partial}{\partial x_l}\sum_{i=l}^{L-1}\mathcal{F}(x_i)\right)$$

The "1+" term ensures gradients never vanish completely.

```
┌──────────────────────────────────────────────────┐
│        RESIDUAL BLOCK (Bottleneck)                 │
├──────────────────────────────────────────────────┤
│                                                    │
│  Input x ─────────────────────────────┐           │
│     │                                  │ (skip)    │
│     ▼                                  │           │
│  ┌──────────────┐                     │           │
│  │ 1×1 Conv     │ (reduce channels)   │           │
│  │ BN + ReLU    │                     │           │
│  └──────┬───────┘                     │           │
│         │                              │           │
│  ┌──────┴───────┐                     │           │
│  │ 3×3 Conv     │ (spatial conv)      │           │
│  │ BN + ReLU    │                     │           │
│  └──────┬───────┘                     │           │
│         │                              │           │
│  ┌──────┴───────┐                     │           │
│  │ 1×1 Conv     │ (expand channels)   │           │
│  │ BN           │                     │           │
│  └──────┬───────┘                     │           │
│         │                              │           │
│         └──────── + ◀─────────────────┘           │
│                   │                                │
│                   ▼                                │
│                 ReLU                               │
│                   │                                │
│                   ▼                                │
│               Output y = F(x) + x                  │
└──────────────────────────────────────────────────┘
```

In [ ]:
class ResidualBlock(nn.Module):
    """Bottleneck residual block."""
    expansion = 4
    
    def __init__(self, in_channels, mid_channels, stride=1, downsample=None):
        super().__init__()
        out_channels = mid_channels * self.expansion
        
        self.conv1 = nn.Conv2d(in_channels, mid_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, out_channels, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out += identity
        return self.relu(out)

class SimpleResNet(nn.Module):
    """Simplified ResNet for ONNX export demonstration."""
    
    def __init__(self, num_classes=1000):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        
        # Layer 1
        downsample1 = nn.Sequential(
            nn.Conv2d(64, 256, 1, bias=False), nn.BatchNorm2d(256))
        self.layer1 = ResidualBlock(64, 64, downsample=downsample1)
        
        # Layer 2 with stride
        downsample2 = nn.Sequential(
            nn.Conv2d(256, 512, 1, stride=2, bias=False), nn.BatchNorm2d(512))
        self.layer2 = ResidualBlock(256, 128, stride=2, downsample=downsample2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        return self.fc(x)

resnet = SimpleResNet(num_classes=10)
resnet.eval()
print(f"SimpleResNet parameters: {sum(p.numel() for p in resnet.parameters()):,}")

# Test forward pass
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = resnet(x)
print(f"Input: {x.shape} → Output: {out.shape}")

## 6. Object Detection: IoU and NMS

### Intersection over Union (IoU)

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

For bounding boxes $A = (x_1^A, y_1^A, x_2^A, y_2^A)$ and $B = (x_1^B, y_1^B, x_2^B, y_2^B)$:

$$x_{\text{inter}} = \max(0, \min(x_2^A, x_2^B) - \max(x_1^A, x_1^B))$$
$$y_{\text{inter}} = \max(0, \min(y_2^A, y_2^B) - \max(y_1^A, y_1^B))$$
$$\text{Area}_{\text{inter}} = x_{\text{inter}} \times y_{\text{inter}}$$

### Non-Maximum Suppression (NMS) Algorithm

```
NMS Algorithm:
─────────────────────────────────────────────
Input:  B = set of boxes, S = scores, τ = IoU threshold
Output: D = set of detections

1. Sort B by scores S in descending order
2. D = ∅
3. While B ≠ ∅:
   a. Select box b* with highest score
   b. D = D ∪ {b*}
   c. Remove b* from B
   d. For each remaining box b in B:
      If IoU(b*, b) > τ:
         Remove b from B
4. Return D
─────────────────────────────────────────────
```

```
Before NMS:                After NMS (τ=0.5):
┌─────────────────────┐   ┌─────────────────────┐
│  ┌───┐              │   │                      │
│  │┌──┼─┐  ┌──┐     │   │  ┌───┐   ┌──┐      │
│  ││  │ │  │  │     │   │  │   │   │  │      │
│  │└──┼─┘  └──┘     │   │  └───┘   └──┘      │
│  └───┘              │   │                      │
│  (overlapping)      │   │  (best per region)  │
└─────────────────────┘   └─────────────────────┘
```

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def nms(boxes, scores, iou_threshold=0.5):
    """Non-Maximum Suppression."""
    indices = np.argsort(scores)[::-1]
    keep = []
    
    while len(indices) > 0:
        current = indices[0]
        keep.append(current)
        
        if len(indices) == 1:
            break
        
        remaining = indices[1:]
        ious = np.array([compute_iou(boxes[current], boxes[i]) for i in remaining])
        indices = remaining[ious <= iou_threshold]
    
    return keep

# Demonstrate NMS
boxes = np.array([
    [100, 100, 210, 210],  # Box A
    [105, 108, 215, 215],  # Box B (overlaps A)
    [110, 105, 220, 218],  # Box C (overlaps A, B)
    [300, 300, 400, 400],  # Box D (isolated)
    [305, 305, 405, 405],  # Box E (overlaps D)
])
scores = np.array([0.9, 0.85, 0.7, 0.95, 0.8])

keep = nms(boxes, scores, iou_threshold=0.5)
print(f"NMS Results:")
print(f"  Input boxes: {len(boxes)}")
print(f"  Kept after NMS: {len(keep)}")
print(f"  Kept indices: {keep}")
print(f"  Kept scores: {scores[keep]}")

# Show IoU matrix
print(f"\nIoU Matrix:")
for i in range(len(boxes)):
    row = [f"{compute_iou(boxes[i], boxes[j]):.3f}" for j in range(len(boxes))]
    print(f"  Box {i}: {' '.join(row)}")

## 7. YOLO Architecture

YOLO (You Only Look Once) divides the image into a grid and predicts
bounding boxes and class probabilities simultaneously:

### YOLO Output Tensor

For each grid cell, YOLO predicts:
$$\text{output} = [t_x, t_y, t_w, t_h, \text{objectness}, c_1, c_2, ..., c_K]$$

Box decoding from anchor boxes:
$$b_x = \sigma(t_x) + c_x$$
$$b_y = \sigma(t_y) + c_y$$
$$b_w = p_w \cdot e^{t_w}$$
$$b_h = p_h \cdot e^{t_h}$$

Where $(c_x, c_y)$ is the grid cell offset, $(p_w, p_h)$ is the anchor size.

```
┌────────────────────────────────────────────────────────┐
│                 YOLO ARCHITECTURE                        │
├────────────────────────────────────────────────────────┤
│                                                          │
│  Input: [B, 3, 640, 640]                                │
│       │                                                  │
│       ▼                                                  │
│  ┌─────────────────────────────────────┐                │
│  │        BACKBONE (CSPDarknet)         │                │
│  │   Feature extraction at 3 scales     │                │
│  └───┬──────────┬──────────────┬───────┘                │
│      │          │              │                         │
│      ▼          ▼              ▼                         │
│  P3: 80×80   P4: 40×40    P5: 20×20                    │
│  (small obj) (med obj)    (large obj)                   │
│      │          │              │                         │
│      ▼          ▼              ▼                         │
│  ┌─────────────────────────────────────┐                │
│  │          NECK (FPN + PAN)            │                │
│  │   Multi-scale feature fusion         │                │
│  └───┬──────────┬──────────────┬───────┘                │
│      │          │              │                         │
│      ▼          ▼              ▼                         │
│  ┌─────────────────────────────────────┐                │
│  │         DETECTION HEAD               │                │
│  │  Per anchor: [x,y,w,h,obj,classes]   │                │
│  └─────────────────────────────────────┘                │
│                                                          │
│  Output: [B, num_predictions, 5+num_classes]            │
│  Total predictions = 80×80×3 + 40×40×3 + 20×20×3       │
│                    = 25,200                               │
└────────────────────────────────────────────────────────┘
```

In [ ]:
class SimplifiedYOLOHead(nn.Module):
    """Simplified YOLO detection head for demonstration."""
    
    def __init__(self, in_channels=256, num_classes=80, num_anchors=3):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors
        out_channels = num_anchors * (5 + num_classes)
        
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, 1)
        )
    
    def forward(self, x):
        B, _, H, W = x.shape
        out = self.conv(x)
        out = out.view(B, self.num_anchors, 5 + self.num_classes, H, W)
        out = out.permute(0, 1, 3, 4, 2)  # [B, anchors, H, W, 5+classes]
        return out

def decode_yolo_output(raw_output, anchors, stride, num_classes=80):
    """Decode raw YOLO output to bounding boxes."""
    B, A, H, W, _ = raw_output.shape
    
    # Grid offsets
    grid_y, grid_x = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    grid_x = grid_x.reshape(1, 1, H, W)
    grid_y = grid_y.reshape(1, 1, H, W)
    
    # Decode boxes
    tx, ty = raw_output[..., 0], raw_output[..., 1]
    tw, th = raw_output[..., 2], raw_output[..., 3]
    obj = raw_output[..., 4]
    cls_scores = raw_output[..., 5:]
    
    # Apply sigmoid to tx, ty, obj
    bx = (1 / (1 + np.exp(-tx)) + grid_x) * stride
    by = (1 / (1 + np.exp(-ty)) + grid_y) * stride
    
    # Apply exp to tw, th with anchors
    anchors = np.array(anchors).reshape(1, A, 1, 1, 2)
    bw = np.exp(tw) * anchors[..., 0] * stride
    bh = np.exp(th) * anchors[..., 1] * stride
    
    objectness = 1 / (1 + np.exp(-obj))
    
    return bx, by, bw, bh, objectness

# Test YOLO head
yolo_head = SimplifiedYOLOHead(in_channels=256, num_classes=80, num_anchors=3)
yolo_head.eval()

feature_map = torch.randn(1, 256, 20, 20)
with torch.no_grad():
    yolo_out = yolo_head(feature_map)

print(f"YOLO Head Output:")
print(f"  Input feature map: {feature_map.shape}")
print(f"  Output shape: {yolo_out.shape}")
print(f"  = [batch, anchors, grid_h, grid_w, 5+classes]")
print(f"  Total predictions: {3 * 20 * 20} = {3*20*20}")

## 8. SSD (Single Shot Detector) Architecture

SSD uses multi-scale feature maps from different backbone layers:

$$\text{Loss} = \frac{1}{N}\left(L_{conf}(x, c) + \alpha \cdot L_{loc}(x, l, g)\right)$$

Where:
- $L_{conf}$ = Cross-entropy loss for classification
- $L_{loc}$ = Smooth L1 loss for box regression
- $N$ = number of matched default boxes

### Default Box (Anchor) Generation

$$s_k = s_{min} + \frac{s_{max} - s_{min}}{m - 1}(k - 1)$$

For aspect ratios $a_r \in \{1, 2, 3, 1/2, 1/3\}$:
$$w_k^a = s_k \sqrt{a_r}, \quad h_k^a = s_k / \sqrt{a_r}$$

In [ ]:
def generate_anchors(feature_sizes, image_size=300, s_min=0.2, s_max=0.9,
                     aspect_ratios=[1, 2, 3, 0.5, 1/3]):
    """Generate SSD-style default anchor boxes."""
    num_layers = len(feature_sizes)
    anchors = []
    
    for k, feat_size in enumerate(feature_sizes):
        s_k = s_min + (s_max - s_min) * k / (num_layers - 1)
        
        for i in range(feat_size):
            for j in range(feat_size):
                cx = (j + 0.5) / feat_size
                cy = (i + 0.5) / feat_size
                
                for ar in aspect_ratios:
                    w = s_k * np.sqrt(ar)
                    h = s_k / np.sqrt(ar)
                    anchors.append([cx, cy, w, h])
    
    return np.array(anchors)

# SSD300 feature map sizes
feature_sizes = [38, 19, 10, 5, 3, 1]
anchors = generate_anchors(feature_sizes)

print(f"SSD Anchor Box Generation")
print("=" * 50)
print(f"  Image size: 300×300")
print(f"  Feature map sizes: {feature_sizes}")
print(f"  Aspect ratios: [1, 2, 3, 1/2, 1/3]")
print(f"  Total anchors: {len(anchors):,}")
print(f"\n  Per scale:")
for k, size in enumerate(feature_sizes):
    count = size * size * 5  # 5 aspect ratios
    print(f"    {size}×{size}: {count:>6} anchors")

## 9. ONNX Export for Object Detection

Exporting detection models requires careful handling of:
1. Multi-scale outputs
2. Post-processing (NMS)
3. Dynamic number of detections

```
ONNX Export Strategy for Detection:
─────────────────────────────────────────
Option A: Include NMS in ONNX graph
  ✓ Single model file
  ✗ Limited NMS operator support
  ✗ Harder to customize

Option B: Export backbone + head only
  ✓ Maximum flexibility
  ✓ Easy to tune NMS threshold
  ✗ Need external post-processing

Recommendation: Option B for production
```

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Export ResNet to ONNX
resnet_onnx_path = 'outputs/resnet_classifier.onnx'
dummy_image = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    resnet,
    dummy_image,
    resnet_onnx_path,
    input_names=['image'],
    output_names=['predictions'],
    dynamic_axes={
        'image': {0: 'batch_size'},
        'predictions': {0: 'batch_size'}
    },
    opset_version=14,
    do_constant_folding=True
)

# Validate
import onnx
model_proto = onnx.load(resnet_onnx_path)
onnx.checker.check_model(model_proto)

file_size = os.path.getsize(resnet_onnx_path) / (1024 * 1024)
print(f"✓ ResNet exported: {file_size:.2f} MB")
print(f"  Nodes: {len(model_proto.graph.node)}")
print(f"  Initializers: {len(model_proto.graph.initializer)}")

# Count op types
ops = {}
for node in model_proto.graph.node:
    ops[node.op_type] = ops.get(node.op_type, 0) + 1
print(f"\n  Operator counts:")
for op, count in sorted(ops.items(), key=lambda x: -x[1]):
    print(f"    {op:<20}: {count}")

## 10. Image Preprocessing in ONNX

Standard ImageNet preprocessing:

$$x_{\text{normalized}} = \frac{x_{\text{pixel}} / 255.0 - \mu}{\sigma}$$

Where $\mu = [0.485, 0.456, 0.406]$ and $\sigma = [0.229, 0.224, 0.225]$.

```
Preprocessing Pipeline:
┌──────────────────────────────────────────────┐
│ Raw Image (H×W×3, uint8, 0-255)             │
│     │                                        │
│     ▼  Resize to 256×256                    │
│     ▼  Center crop to 224×224               │
│     ▼  Convert to float32, divide by 255    │
│     ▼  Normalize (subtract mean, div std)   │
│     ▼  Transpose HWC → CHW                  │
│     ▼  Add batch dimension                  │
│                                              │
│ Model Input (1×3×224×224, float32)          │
└──────────────────────────────────────────────┘
```

In [ ]:
import onnxruntime as ort

def preprocess_image(image_array, target_size=224):
    """Standard ImageNet preprocessing."""
    # Simulate resize (in practice use PIL/cv2)
    H, W, C = image_array.shape
    
    # Simple resize by taking every nth pixel
    scale = max(H, W) / 256
    new_h, new_w = int(H / scale), int(W / scale)
    
    # Center crop
    start_h = (new_h - target_size) // 2
    start_w = (new_w - target_size) // 2
    
    # Normalize
    img = image_array[:target_size, :target_size, :].astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = (img - mean) / std
    
    # HWC → CHW
    img = img.transpose(2, 0, 1)
    
    # Add batch dimension
    return img[np.newaxis, ...].astype(np.float32)

# Simulate an image
fake_image = np.random.randint(0, 256, (300, 400, 3), dtype=np.uint8)
processed = preprocess_image(fake_image)
print(f"Raw image: {fake_image.shape} (H, W, C) uint8")
print(f"Processed: {processed.shape} (B, C, H, W) float32")
print(f"Value range: [{processed.min():.2f}, {processed.max():.2f}]")

# Run inference
session = ort.InferenceSession(resnet_onnx_path)
output = session.run(None, {'image': processed})[0]
print(f"\nInference output: {output.shape}")
predicted_class = np.argmax(output[0])
confidence = np.exp(output[0]) / np.exp(output[0]).sum()  # Softmax
print(f"Predicted class: {predicted_class} (confidence: {confidence[predicted_class]:.4f})")

## 11. Feature Pyramid Network (FPN)

FPN enables multi-scale detection by combining low-resolution, semantically strong
features with high-resolution, spatially precise features:

$$P_l = \text{Conv}_{1\times1}(C_l) + \text{Upsample}(P_{l+1})$$

```
┌──────────────────────────────────────────────────────┐
│              FEATURE PYRAMID NETWORK                   │
├──────────────────────────────────────────────────────┤
│                                                        │
│  Backbone         Lateral          Top-down + Output  │
│  (bottom-up)      (1×1 conv)       (merge + 3×3 conv)│
│                                                        │
│  C2: 256×256 ──▶ ──────────────────────────▶ P2      │
│       │                                   ▲           │
│       ▼                                   │ upsample  │
│  C3: 128×128 ──▶ ─────────── + ──────────▶ P3       │
│       │                        ▲                      │
│       ▼                        │ upsample             │
│  C4: 64×64   ──▶ ──── + ─────▶ P4                   │
│       │               ▲                               │
│       ▼               │ upsample                      │
│  C5: 32×32   ──▶ ────▶ P5                            │
│                                                        │
│  Semantics:  weak ←──────────────────▶ strong        │
│  Resolution: high  ←──────────────────▶ low          │
└──────────────────────────────────────────────────────┘
```

In [ ]:
class SimpleFPN(nn.Module):
    """Simplified Feature Pyramid Network."""
    
    def __init__(self, in_channels_list=[64, 128, 256, 512], out_channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, 1) for in_ch in in_channels_list
        ])
        self.output_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels_list
        ])
    
    def forward(self, features):
        # Lateral connections
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        
        # Top-down pathway
        for i in range(len(laterals) - 2, -1, -1):
            upsampled = nn.functional.interpolate(
                laterals[i + 1], size=laterals[i].shape[2:], mode='nearest')
            laterals[i] = laterals[i] + upsampled
        
        # Output convolutions
        outputs = [conv(lat) for conv, lat in zip(self.output_convs, laterals)]
        return outputs

# Test FPN
fpn = SimpleFPN()
fpn.eval()

# Simulate backbone features at different scales
features = [
    torch.randn(1, 64, 56, 56),   # C2
    torch.randn(1, 128, 28, 28),  # C3
    torch.randn(1, 256, 14, 14),  # C4
    torch.randn(1, 512, 7, 7),    # C5
]

with torch.no_grad():
    pyramid = fpn(features)

print("Feature Pyramid Network Output:")
print("=" * 50)
for i, (inp, out) in enumerate(zip(features, pyramid)):
    print(f"  C{i+2}: {list(inp.shape):>20} → P{i+2}: {list(out.shape)}")

## 12. Data Augmentation and Its ONNX Implications

Training augmentations that affect ONNX export:

| Augmentation | Training | ONNX Inference | Notes |
|-------------|----------|----------------|-------|
| Random Crop | Yes | No (center crop) | Fixed size input |
| Random Flip | Yes | No | Deterministic |
| Color Jitter | Yes | No | Preprocessing only |
| Mosaic | Yes | No | YOLO-specific |
| Test-Time Aug (TTA) | - | Optional | Multiple passes |

### Test-Time Augmentation

$$\hat{y} = \frac{1}{|\mathcal{A}|}\sum_{a \in \mathcal{A}} f(a(x))$$

Common TTA: original + horizontal flip, multi-scale

In [ ]:
def test_time_augmentation(session, image, scales=[0.8, 1.0, 1.2]):
    """Perform test-time augmentation with ONNX model."""
    all_predictions = []
    B, C, H, W = image.shape
    
    for scale in scales:
        # Scale
        new_h, new_w = int(H * scale), int(W * scale)
        # In practice, use proper interpolation
        scaled = np.random.randn(B, C, 224, 224).astype(np.float32)  # Simulated
        
        # Original
        pred = session.run(None, {'image': scaled})[0]
        all_predictions.append(pred)
        
        # Horizontal flip
        flipped = scaled[:, :, :, ::-1].copy()
        pred_flip = session.run(None, {'image': flipped})[0]
        all_predictions.append(pred_flip)
    
    # Average predictions
    avg_pred = np.mean(all_predictions, axis=0)
    return avg_pred

# Run TTA
test_image = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Single pass
single_pred = session.run(None, {'image': test_image})[0]

# TTA
tta_pred = test_time_augmentation(session, test_image)

print(f"Test-Time Augmentation Results:")
print(f"  Single pass prediction: class {np.argmax(single_pred[0])}")
print(f"  TTA prediction: class {np.argmax(tta_pred[0])}")
print(f"  TTA uses {3 * 2} forward passes (3 scales × 2 flips)")
print(f"  Confidence boost: {np.max(np.exp(tta_pred[0])/np.exp(tta_pred[0]).sum()) - np.max(np.exp(single_pred[0])/np.exp(single_pred[0]).sum()):.4f}")

## 13. Quantization for CV Models

CV models are generally more tolerant to quantization than NLP models:

$$\text{Quantize}(x) = \text{clip}\left(\text{round}\left(\frac{x}{s}\right) + z,\; 0,\; 255\right)$$

### Per-Channel vs Per-Tensor Quantization

**Per-tensor**: One scale/zero-point for the entire tensor
$$s = \frac{\max(|W|)}{127}$$

**Per-channel**: One scale/zero-point per output channel
$$s_c = \frac{\max(|W_c|)}{127}$$

Per-channel is better for Conv layers where weight distributions vary across filters.

### Typical Accuracy Results

| Model | FP32 Top-1 | INT8 Top-1 | Drop |
|-------|-----------|-----------|------|
| ResNet-50 | 76.1% | 75.8% | 0.3% |
| MobileNetV2 | 72.0% | 71.2% | 0.8% |
| EfficientNet-B0 | 77.1% | 76.5% | 0.6% |
| YOLOv5s (mAP) | 37.4 | 36.9 | 0.5 |

In [ ]:
from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType
from onnxruntime.quantization import CalibrationDataReader
import time

# Dynamic quantization
resnet_int8_path = 'outputs/resnet_int8.onnx'
quantize_dynamic(
    resnet_onnx_path,
    resnet_int8_path,
    weight_type=QuantType.QInt8
)

# Compare sizes and speed
fp32_size = os.path.getsize(resnet_onnx_path) / (1024 * 1024)
int8_size = os.path.getsize(resnet_int8_path) / (1024 * 1024)

print(f"Quantization Results:")
print(f"  FP32 size: {fp32_size:.2f} MB")
print(f"  INT8 size: {int8_size:.2f} MB")
print(f"  Compression: {fp32_size/int8_size:.2f}×")

# Benchmark
fp32_sess = ort.InferenceSession(resnet_onnx_path)
int8_sess = ort.InferenceSession(resnet_int8_path)

test_input = {'image': np.random.randn(1, 3, 224, 224).astype(np.float32)}

def bench(sess, n=50):
    for _ in range(10):  # warmup
        sess.run(None, test_input)
    start = time.perf_counter()
    for _ in range(n):
        sess.run(None, test_input)
    return (time.perf_counter() - start) / n * 1000

fp32_ms = bench(fp32_sess)
int8_ms = bench(int8_sess)
print(f"\n  FP32 latency: {fp32_ms:.2f} ms")
print(f"  INT8 latency: {int8_ms:.2f} ms")
print(f"  Speedup: {fp32_ms/int8_ms:.2f}×")

## 14. Detection Model Post-Processing Pipeline

Complete detection pipeline outside the ONNX graph:

```
┌────────────────────────────────────────────────────────────┐
│          DETECTION POST-PROCESSING                          │
├────────────────────────────────────────────────────────────┤
│                                                              │
│  Raw Model Output: [B, num_predictions, 5+classes]          │
│       │                                                      │
│       ▼                                                      │
│  1. DECODE BOXES                                            │
│     • Apply sigmoid to tx, ty, objectness                   │
│     • Apply exp to tw, th                                   │
│     • Convert to absolute coordinates                       │
│       │                                                      │
│       ▼                                                      │
│  2. FILTER BY CONFIDENCE                                    │
│     • score = objectness × class_prob                       │
│     • Keep if score > threshold (e.g., 0.25)               │
│       │                                                      │
│       ▼                                                      │
│  3. PER-CLASS NMS                                           │
│     • For each class independently                          │
│     • IoU threshold (e.g., 0.45)                            │
│       │                                                      │
│       ▼                                                      │
│  4. TOP-K SELECTION                                         │
│     • Keep top 100-300 detections                           │
│       │                                                      │
│       ▼                                                      │
│  Final: List of [x1, y1, x2, y2, score, class_id]         │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
def detection_postprocess(raw_output, conf_threshold=0.25, iou_threshold=0.45,
                          max_detections=300):
    """Complete detection post-processing pipeline."""
    # raw_output: [batch, num_preds, 5+num_classes]
    batch_size = raw_output.shape[0]
    results = []
    
    for b in range(batch_size):
        preds = raw_output[b]  # [num_preds, 5+classes]
        
        # Extract components
        boxes_xywh = preds[:, :4]
        objectness = 1 / (1 + np.exp(-preds[:, 4]))  # sigmoid
        class_probs = 1 / (1 + np.exp(-preds[:, 5:]))  # sigmoid
        
        # Compute scores: objectness × class_prob
        scores = objectness[:, None] * class_probs
        
        # Find best class per prediction
        class_ids = np.argmax(scores, axis=1)
        max_scores = np.max(scores, axis=1)
        
        # Filter by confidence
        mask = max_scores > conf_threshold
        filtered_boxes = boxes_xywh[mask]
        filtered_scores = max_scores[mask]
        filtered_classes = class_ids[mask]
        
        # Convert xywh to xyxy
        if len(filtered_boxes) > 0:
            x1 = filtered_boxes[:, 0] - filtered_boxes[:, 2] / 2
            y1 = filtered_boxes[:, 1] - filtered_boxes[:, 3] / 2
            x2 = filtered_boxes[:, 0] + filtered_boxes[:, 2] / 2
            y2 = filtered_boxes[:, 1] + filtered_boxes[:, 3] / 2
            boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1)
            
            # Apply NMS per class
            keep_indices = nms(boxes_xyxy, filtered_scores, iou_threshold)
            keep_indices = keep_indices[:max_detections]
            
            results.append({
                'boxes': boxes_xyxy[keep_indices],
                'scores': filtered_scores[keep_indices],
                'class_ids': filtered_classes[keep_indices]
            })
        else:
            results.append({'boxes': np.array([]), 'scores': np.array([]), 'class_ids': np.array([])})
    
    return results

# Simulate detection output
num_preds = 8400  # Typical YOLO output
num_classes = 80
raw_detections = np.random.randn(1, num_preds, 5 + num_classes).astype(np.float32)

results = detection_postprocess(raw_detections, conf_threshold=0.5)
print(f"Detection Post-Processing:")
print(f"  Raw predictions: {num_preds}")
print(f"  After filtering: {len(results[0]['scores'])} detections")
if len(results[0]['scores']) > 0:
    print(f"  Top score: {results[0]['scores'][0]:.4f}")
    print(f"  Class distribution: {np.bincount(results[0]['class_ids'].astype(int), minlength=5)[:5]}...")

## 15. Inference Optimization: IO Binding

For GPU inference, IO Binding avoids CPU↔GPU memory copies:

```
Without IO Binding:           With IO Binding:
┌─────┐  copy  ┌─────┐      ┌─────┐         ┌─────┐
│ CPU │ ────▶ │ GPU │      │ GPU │ ───────▶ │ GPU │
│input│        │model│      │input│  (zero   │model│
└─────┘        └──┬──┘      └─────┘   copy)  └──┬──┘
                  │                              │
        ┌─────┐  │ copy              ┌─────┐   │
        │ CPU │ ◀┘                   │ GPU │ ◀─┘
        │ out │                      │ out │
        └─────┘                      └─────┘
  2 memory copies                0 memory copies
  (adds latency)                (minimum latency)
```

In [ ]:
# Demonstrate session options for CV models
def create_optimized_session(model_path, use_gpu=False):
    """Create an optimized ONNX Runtime session for CV inference."""
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 4
    opts.inter_op_num_threads = 1
    opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    providers = ['CPUExecutionProvider']
    if use_gpu and 'CUDAExecutionProvider' in ort.get_available_providers():
        providers = ['CUDAExecutionProvider'] + providers
    
    return ort.InferenceSession(model_path, opts, providers=providers)

# Benchmark batch processing
opt_session = create_optimized_session(resnet_onnx_path)

print("CV Inference Benchmark")
print("=" * 55)
print(f"{'Batch Size':>10} {'Total (ms)':>12} {'Per Image (ms)':>14} {'Images/sec':>12}")
print("-" * 55)

for batch_size in [1, 2, 4, 8, 16]:
    input_data = {'image': np.random.randn(batch_size, 3, 224, 224).astype(np.float32)}
    
    # Warmup
    for _ in range(5):
        opt_session.run(None, input_data)
    
    # Measure
    times = []
    for _ in range(20):
        start = time.perf_counter()
        opt_session.run(None, input_data)
        times.append((time.perf_counter() - start) * 1000)
    
    mean_ms = np.mean(times)
    per_image = mean_ms / batch_size
    throughput = batch_size / (mean_ms / 1000)
    print(f"{batch_size:>10} {mean_ms:>12.2f} {per_image:>14.2f} {throughput:>12.1f}")

## 16. Model Architectures Comparison

### Accuracy vs Speed Trade-off

```
Accuracy (Top-1 %) ▲
                   │
        85% ─      │              ● EfficientNet-B7
                   │          ● EfficientNet-B4
        80% ─      │      ● ResNet-152
                   │    ● ResNet-50    ● EfficientNet-B0
        75% ─      │  ● ResNet-18
                   │● MobileNetV2
        70% ─      │
                   │
        65% ─      │● MobileNetV1
                   │
                   └──────────────────────────────────▶
                   1ms    5ms   10ms   50ms  100ms
                            Inference Latency
```

### Choosing the Right Model for ONNX Deployment

| Use Case | Recommended | Why |
|----------|-------------|-----|
| Edge/Mobile | MobileNetV3, EfficientNet-Lite | Small, fast |
| Server (latency) | ResNet-50, EfficientNet-B0 | Good accuracy/speed |
| Server (accuracy) | EfficientNet-B4+ | Best accuracy |
| Real-time detection | YOLOv5s/v8n | Optimized for speed |
| High-accuracy detection | YOLOv5x, DETR | Best mAP |

In [ ]:
# Model comparison table
models_comparison = [
    ('ResNet-18', 11.7, 1.8, 69.8, 4.5),
    ('ResNet-50', 25.6, 4.1, 76.1, 8.2),
    ('ResNet-152', 60.2, 11.6, 78.3, 22.1),
    ('MobileNetV2', 3.4, 0.3, 72.0, 2.1),
    ('MobileNetV3-L', 5.4, 0.2, 75.2, 2.8),
    ('EfficientNet-B0', 5.3, 0.4, 77.1, 4.8),
    ('EfficientNet-B4', 19.0, 4.2, 82.9, 18.5),
    ('YOLOv5s (det)', 7.2, 16.5, 37.4, 3.2),  # mAP instead of Top-1
]

print("\nCV Model Comparison for ONNX Deployment")
print("=" * 70)
print(f"{'Model':<18} {'Params (M)':>10} {'GFLOPs':>8} {'Accuracy':>10} {'Lat (ms)':>10}")
print("-" * 70)
for name, params, flops, acc, lat in models_comparison:
    acc_str = f"{acc:.1f}%" if acc > 50 else f"{acc:.1f} mAP"
    print(f"{name:<18} {params:>10.1f} {flops:>8.1f} {acc_str:>10} {lat:>10.1f}")

print("\n  * Latency measured on CPU with ONNX Runtime (batch=1, 224×224)")
print("  * YOLOv5s uses mAP@0.5 on COCO instead of ImageNet Top-1")

## 17. ONNX Operator Patterns for CV

### Common CV operators in ONNX:

| Operation | ONNX Op | Notes |
|-----------|---------|-------|
| Convolution | Conv | Most FLOPs |
| BatchNorm | BatchNormalization | Fuses with Conv |
| ReLU | Relu | Zero-cost with fusion |
| Max Pooling | MaxPool | Spatial reduction |
| Avg Pooling | AveragePool / GlobalAveragePool | |
| Upsample | Resize | FPN, decoder |
| Concat | Concat | Feature fusion |
| Add | Add | Skip connections |

### Fusion Opportunities

$$\text{Conv} + \text{BN} + \text{ReLU} \rightarrow \text{FusedConvBNReLU}$$

This is the most impactful optimization for CV models,
reducing memory bandwidth by ~3× for these layers.

In [ ]:
# Analyze operator patterns in our ResNet model
model = onnx.load(resnet_onnx_path)

# Find fusible patterns
def find_fusion_patterns(graph):
    """Find Conv+BN and Conv+BN+Relu patterns."""
    patterns = {'conv_bn': 0, 'conv_bn_relu': 0, 'add_relu': 0}
    
    node_output_map = {}
    for node in graph.node:
        for output in node.output:
            node_output_map[output] = node
    
    for node in graph.node:
        if node.op_type == 'BatchNormalization':
            # Check if input comes from Conv
            if node.input[0] in node_output_map:
                prev = node_output_map[node.input[0]]
                if prev.op_type == 'Conv':
                    patterns['conv_bn'] += 1
        
        if node.op_type == 'Relu':
            if node.input[0] in node_output_map:
                prev = node_output_map[node.input[0]]
                if prev.op_type == 'BatchNormalization':
                    patterns['conv_bn_relu'] += 1
                elif prev.op_type == 'Add':
                    patterns['add_relu'] += 1
    
    return patterns

patterns = find_fusion_patterns(model.graph)
print("Fusible Patterns Found:")
print("=" * 40)
for pattern, count in patterns.items():
    print(f"  {pattern}: {count}")
print(f"\nTotal potential fusions: {sum(patterns.values())}")
print(f"This would reduce node count by ~{sum(patterns.values()) * 2}")

## 18. Summary and Best Practices

### CV + ONNX Key Points

1. **Conv+BN Fusion**: Always happens automatically, massive speed gain
2. **Quantization**: CV models tolerate INT8 well (~0.5% accuracy drop)
3. **Input Preprocessing**: Keep outside ONNX graph for flexibility
4. **Detection Post-Processing**: NMS outside graph for tunability
5. **Batch Processing**: Leverage batch dimension for throughput
6. **IO Binding**: Essential for GPU deployment

### Performance Summary

```
┌──────────────────────────────────────────────────────────┐
│     CV MODEL DEPLOYMENT DECISION TREE                     │
├──────────────────────────────────────────────────────────┤
│                                                            │
│  Is latency critical? (< 10ms)                            │
│    YES → MobileNet/EfficientNet-Lite + INT8              │
│    NO  → ResNet-50/EfficientNet-B4 + FP16/INT8           │
│                                                            │
│  Is accuracy critical? (> 80% Top-1)                      │
│    YES → EfficientNet-B4+ + FP32/FP16                    │
│    NO  → Smaller model + aggressive quantization         │
│                                                            │
│  Target hardware?                                         │
│    CPU  → INT8 quantization + thread tuning              │
│    GPU  → FP16 + TensorRT EP + IO Binding                │
│    Edge → NNAPI/CoreML EP + INT8                         │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# Final summary
print("""
╔═══════════════════════════════════════════════════════════════╗
║          CV + ONNX DEEP DIVE COMPLETE                        ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  Topics Covered:                                              ║
║  • Conv2D mathematics and FLOPs analysis                     ║
║  • Batch Normalization fusion                                 ║
║  • ResNet architecture and skip connections                  ║
║  • IoU computation and NMS algorithm                         ║
║  • YOLO detection architecture                                ║
║  • SSD and anchor box generation                             ║
║  • Feature Pyramid Networks                                   ║
║  • Quantization for CV models                                ║
║  • Post-processing pipelines                                 ║
║  • Performance optimization and benchmarking                 ║
║                                                               ║
║  Key Formula: IoU = |A ∩ B| / |A ∪ B|                       ║
║                                                               ║
║  Key Insight: CV models benefit greatly from:                ║
║    1. Conv+BN fusion (automatic in ORT)                      ║
║    2. INT8 quantization (<1% accuracy drop)                  ║
║    3. Batch processing for throughput                        ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")